In [ ]:
import os
import PIL
from tqdm import tqdm
import torch
import torchvision
import torchvision.transforms as transforms

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Assuming that we are on a CUDA machine, this should print a CUDA device:

print(device)

In [ ]:
def get_folder_structure(base = 'data'):
    """
    Returns the folder structure of the data folder
    """
    return [x[0] for x in os.walk(base)]
get_folder_structure()

In [ ]:
transform = transforms.Compose(
    [transforms.Resize((512, 512)),
     transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 32

In [ ]:
# Load the training data from the data folder
trainset = torchvision.datasets.ImageFolder(root='data/train/', transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)
testset = torchvision.datasets.ImageFolder(root='data/test/', transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('0-noDR', '1-mild', '2-moderate', '3-severe', '4-proliferativeDR')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# functions to show an image


def imshow(img):
    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()


# get some random training images
dataiter = iter(trainloader)
images, labels = next(dataiter)

# show images
imshow(torchvision.utils.make_grid(images))
# print labels
print(' '.join(f'{classes[labels[j]]:5s}' for j in range(batch_size)))

In [ ]:
# Cargar un modelo preentrenado
model = torchvision.models.resnet50(pretrained=True)

# Ajustar la capa final para tu número de clases (puedes cambiar este ejemplo si hay más de 2 clases)
num_classes = 5  # Cambia a tu número de clases si es diferente
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)

# Llevar el modelo a GPU si está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
criterion = torch.nn.CrossEntropyLoss()  # Usa CrossEntropy para clasificación múltiple
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
num_epochs = 10  # Ajusta como sea necesario

for epoch in range(num_epochs):
    model.train()  # Establece el modo de entrenamiento
    running_loss = 0.0

    for images, labels in tqdm(trainloader):
        images, labels = images.to(device), labels.to(device)  # Mueve a GPU si es necesario

        optimizer.zero_grad()  # Limpia los gradientes

        outputs = model(images)  # Propagación hacia adelante

        loss = criterion(outputs, labels)  # Calcular la pérdida
        loss.backward()  # Propagación hacia atrás
        optimizer.step()  # Actualizar pesos

        running_loss += loss.item()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(trainloader):.4f}')

In [ ]:
model.eval()  # Establece modo de evaluación
correct = 0
total = 0

with torch.no_grad():  # Desactiva el cálculo de gradientes para evaluación
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Precisión en el conjunto de prueba: {100 * correct / total:.2f}%')